## Task 1: Identify and Characterize Data Sources

**Step 1:** Create the following three datasets in memory as Python data structures (lists of dictionaries):

**Dataset A** — A list of 8 user registration entries with fields: `username`, `email`, `age`, and `signup_date`. Include at least 2 entries with problems: one with an invalid email (missing `@`), one with age as a string instead of a number, and one with an empty username.

**Dataset B** — A list of 10 application log entries with fields: `timestamp`, `service`, `level` (INFO/WARNING/ERROR), and `message`. Include entries from at least 3 different services.

**Dataset C** — A list of 6 product inventory records with fields: `sku`, `product_name`, `quantity_in_stock`, `warehouse_location`, and `last_updated`.

In [1]:
import pandas as pd
import numpy as np

# Dataset A: User Registrations (8 entries)
dataset_a = [
    {"username": "alice_99", "email": "alice@example.com", "age": 25, "signup_date": "2025-01-01"},
    {"username": "bob_builder", "email": "bob_at_work.com", "age": 42, "signup_date": "2025-01-02"}, # Invalid email (missing @)
    {"username": "charlie_b", "email": "charlie@peanuts.com", "age": "30", "signup_date": "2025-01-03"}, # Age as string
    {"username": "", "email": "ghost@void.com", "age": 19, "signup_date": "2025-01-04"}, # Empty username
    {"username": "dana_v", "email": "dana@web.com", "age": 22, "signup_date": "2025-01-05"},
    {"username": "eve_online", "email": "eve@server.com", "age": -5, "signup_date": "2025-01-06"}, # Invalid age logic
    {"username": "frank_castle", "email": "punisher@marvel.com", "age": 35, "signup_date": "2025-01-07"},
    {"username": "grace_h", "email": "grace@hopper.org", "age": 80, "signup_date": "2025-01-08"}
]

# Dataset B: Application Log Entries (10 entries from 3 services)
dataset_b = [
    {"timestamp": "2025-03-01 10:00:01", "service": "auth-api", "level": "INFO", "message": "User login successful"},
    {"timestamp": "2025-03-01 10:00:05", "service": "payment-gateway", "level": "ERROR", "message": "Timeout connecting to bank"},
    {"timestamp": "2025-03-01 10:01:12", "service": "auth-api", "level": "WARNING", "message": "Failed login attempt"},
    {"timestamp": "2025-03-01 10:02:45", "service": "inventory-mgmt", "level": "INFO", "message": "Stock updated for SKU-102"},
    {"timestamp": "2025-03-01 10:03:01", "service": "payment-gateway", "level": "INFO", "message": "Transaction ORD-99 completed"},
    {"timestamp": "2025-03-01 10:04:20", "service": "inventory-mgmt", "level": "ERROR", "message": "Database connection lost"},
    {"timestamp": "2025-03-01 10:05:10", "service": "auth-api", "level": "INFO", "message": "Session token refreshed"},
    {"timestamp": "2025-03-01 10:06:15", "service": "payment-gateway", "level": "WARNING", "message": "Latency spike detected"},
    {"timestamp": "2025-03-01 10:07:00", "service": "inventory-mgmt", "level": "INFO", "message": "Cron job started"},
    {"timestamp": "2025-03-01 10:08:30", "service": "auth-api", "level": "ERROR", "message": "Unhandled exception in middleware"}
]

# Dataset C: Product Inventory Records (6 entries)
dataset_c = [
    {"sku": "SKU-001", "product_name": "Mechanical Keyboard", "quantity_in_stock": 150, "warehouse_location": "Berlin-01", "last_updated": "2025-02-28"},
    {"sku": "SKU-002", "product_name": "Gaming Mouse", "quantity_in_stock": 300, "warehouse_location": "Seoul-05", "last_updated": "2025-02-27"},
    {"sku": "SKU-003", "product_name": "UltraWide Monitor", "quantity_in_stock": 45, "warehouse_location": "Nairobi-02", "last_updated": "2025-02-26"},
    {"sku": "SKU-004", "product_name": "USB-C Hub", "quantity_in_stock": 1000, "warehouse_location": "Toronto-01", "last_updated": "2025-02-25"},
    {"sku": "SKU-005", "product_name": "Desk Lamp", "quantity_in_stock": 85, "warehouse_location": "Lima-03", "last_updated": "2025-02-24"},
    {"sku": "SKU-006", "product_name": "Noise Cancelling Headphones", "quantity_in_stock": 120, "warehouse_location": "Berlin-01", "last_updated": "2025-02-23"}
]

**Step 2:** For each dataset, write a short markdown paragraph answering:

* What type of data source does this represent (user input, system-generated, internal database)?
* What are the likely quality issues you would expect from this source in production?
* What kind of validation would you apply before using this data in an ML pipeline?

#### **Dataset A: User Registration Entries**

* **Data Source Type:** User Input.
* **Likely Quality Issues:** This source is highly prone to human error. Typical issues include malformed email addresses, empty required fields (like usernames), inconsistent data types (e.g., age as a string), and logically impossible values (e.g., negative ages).
* **Validation Strategy:** Before using this in an ML pipeline, I would apply strict schema validation to ensure types are correct, use Regular Expressions (Regex) to verify email formats, and implement range checks to ensure numeric values are within a realistic domain.

#### **Dataset B: Application Log Entries**

* **Data Source Type:** System-Generated Data.
* **Likely Quality Issues:** Issues often involve high-volume redundancy, inconsistent timestamps if servers are in different time zones, or "missing" logs if a crash prevents the system from writing to the disk.
* **Validation Strategy:** I would parse the timestamps into standard datetime objects, normalize the service names, and filter entries based on severity levels (INFO/WARNING/ERROR) to isolate significant events.

#### **Dataset C: Product Inventory Records**

* **Data Source Type:** Internal Database.
* **Likely Quality Issues:** The main risk is "stale" data where the `quantity_in_stock` doesn't reflect real-time sales due to synchronization lag. Duplicate SKUs or inconsistent warehouse location naming conventions are also common.
* **Validation Strategy:** Validation would include checking for unique SKU constraints, verifying that quantities are non-negative, and performing "freshness" checks to ensure the `last_updated` field is within an acceptable window.

**Step 3:** Write a validation function called `validate_registrations(entries)` that takes Dataset A and returns two lists: `valid_entries` and `invalid_entries`. A valid entry must have a non-empty username, an email containing `@`, and an age that can be converted to a positive integer. Test your function on Dataset A and print the counts.

In [2]:
def validate_registrations(entries):
    valid_entries = []
    invalid_entries = []
    
    for entry in entries:
        username = entry.get('username', "")
        email = entry.get('email', "")
        age = entry.get('age')
        
        # Validation Logic
        is_user_valid = bool(username and username.strip())
        is_email_valid = "@" in email
        
        # Try to convert age to positive integer
        try:
            age_val = int(age)
            is_age_valid = age_val > 0
        except (ValueError, TypeError):
            is_age_valid = False
            
        if is_user_valid and is_email_valid and is_age_valid:
            valid_entries.append(entry)
        else:
            invalid_entries.append(entry)
            
    return valid_entries, invalid_entries

# Testing the function
valid_list, invalid_list = validate_registrations(dataset_a)

print(f"Total entries: {len(dataset_a)}")
print(f"Valid entries count: {len(valid_list)}")
print(f"Invalid entries count: {len(invalid_list)}")

# Optional: Display results to verify
print("\nInvalid Entries found:")
for item in invalid_list:
    print(item)

Total entries: 8
Valid entries count: 5
Invalid entries count: 3

Invalid Entries found:
{'username': 'bob_builder', 'email': 'bob_at_work.com', 'age': 42, 'signup_date': '2025-01-02'}
{'username': '', 'email': 'ghost@void.com', 'age': 19, 'signup_date': '2025-01-04'}
{'username': 'eve_online', 'email': 'eve@server.com', 'age': -5, 'signup_date': '2025-01-06'}


## Task 2: Work with Multiple Data Formats

**Step 1:** Create a pandas DataFrame called `rides` with 500 rows and the following columns:

* `ride_id`: integers from 1 to 500

* `driver_id`: random integers between 1000 and 1050

* `distance_km`: random floats between 1.0 and 30.0

* `fare_usd`: random floats between 5.0 and 75.0

* `ride_type`: randomly chosen from `["standard", "premium", "pool"]`

* `city`: randomly chosen from `["Berlin", "Seoul", "Nairobi", "Toronto", "Lima"]`

* `timestamp`: a date range starting from `"2025-01-01"` with minute frequency

In [3]:
import pandas as pd
import numpy as np

# Optional: Seed for reproducibility
np.random.seed(42)

# Generate 500 rows of ride data
rides = pd.DataFrame({
    "ride_id": range(1, 501),
    "driver_id": np.random.randint(1000, 1051, 500),
    "distance_km": np.random.uniform(1.0, 30.0, 500),
    "fare_usd": np.random.uniform(5.0, 75.0, 500),
    "ride_type": np.random.choice(["standard", "premium", "pool"], 500),
    "city": np.random.choice(["Berlin", "Seoul", "Nairobi", "Toronto", "Lima"], 500),
    "timestamp": pd.date_range(start="2025-01-01", periods=500, freq="min")
})

# Display the first few rows to verify
rides.head()

,ride_id,driver_id,distance_km,fare_usd,ride_type,city,timestamp
0,1,1038,3.662173,45.434237,standard,Berlin,2025-01-01 00:00:00
1,2,1028,3.730553,16.585932,pool,Seoul,2025-01-01 00:01:00
2,3,1014,10.030986,7.371541,pool,Seoul,2025-01-01 00:02:00
3,4,1042,29.405805,26.803122,premium,Lima,2025-01-01 00:03:00
4,5,1007,6.084578,59.634700,pool,Nairobi,2025-01-01 00:04:00


**Step 2:** Save the DataFrame in three formats:

* CSV: `rides.csv`
* JSON: `rides.json` (using `orient="records"` and `lines=True`)
* Parquet: `rides.parquet`

In [4]:
# 1. Save as CSV
rides.to_csv("rides.csv", index=False)

# 2. Save as JSON
rides.to_json("rides.json", orient="records", lines=True, date_format="iso")

# 3. Save as Parquet
rides.to_parquet("rides.parquet", engine="pyarrow")

print("Files saved successfully: rides.csv, rides.json, rides.parquet")

Files saved successfully: rides.csv, rides.json, rides.parquet


**Step 3:** Compare the file sizes of all three formats. Print the size of each file in KB and compute how many times smaller Parquet is compared to CSV.

In [5]:
import os

# Helper function to get size in KB
def get_file_size_kb(file_path):
    size_bytes = os.path.getsize(file_path)
    return size_bytes / 1024

# Measure sizes
csv_kb = get_file_size_kb("rides.csv")
json_kb = get_file_size_kb("rides.json")
parquet_kb = get_file_size_kb("rides.parquet")

# Calculate ratio
reduction_ratio = csv_kb / parquet_kb

print(f"File Size Comparison:")
print(f"- CSV:     {csv_kb:.2f} KB")
print(f"- JSON:    {json_kb:.2f} KB")
print(f"- Parquet: {parquet_kb:.2f} KB")
print(f"\nParquet is {reduction_ratio:.2f} times smaller than CSV.")

File Size Comparison:
- CSV:     39.49 KB
- JSON:    78.30 KB
- Parquet: 20.76 KB

Parquet is 1.90 times smaller than CSV.


**Step 4:** Read each file back into a new DataFrame and verify that the data is identical by comparing shapes and a sample of values. Note any differences you observe (for example, type changes after reading from CSV).

In [6]:
# Read files back
df_csv = pd.read_csv("rides.csv")
df_json = pd.read_json("rides.json", orient="records", lines=True)
df_parquet = pd.read_parquet("rides.parquet")

print(f"CSV Shape: {df_csv.shape}")
print(f"JSON Shape: {df_json.shape}")
print(f"Parquet Shape: {df_parquet.shape}")

# Check dtypes for the 'timestamp' column specifically
print("\nTimestamp Data Types:")
print(f"Original: {rides['timestamp'].dtype}")
print(f"From CSV: {df_csv['timestamp'].dtype}")
print(f"From Parquet: {df_parquet['timestamp'].dtype}")

CSV Shape: (500, 7)
JSON Shape: (500, 7)
Parquet Shape: (500, 7)

Timestamp Data Types:
Original: datetime64[us]
From CSV: str
From Parquet: datetime64[us]


**Step 5:** Write a markdown cell summarizing: Which format preserved types most faithfully? Which was smallest? When would you choose each format?

**Which format preserved types most faithfully?**
**Parquet** preserved types most faithfully. It kept the `timestamp` column as a `datetime64` object. In contrast, the CSV format converted the timestamps into strings (`object`), requiring an extra conversion step after loading.

**Which was smallest?**
**Parquet** was the smallest. This is because Parquet is a binary format that uses efficient encoding and compression, whereas CSV and JSON are text-based and repeat column names or delimiters for every single row.

**When would you choose each format?**

* **CSV:** Choose for small datasets that need to be opened in Excel or shared with people who don't use Python/Programming tools.
* **JSON:** Choose for web applications or when data is "nested" (like a list of items inside a single column).
* **Parquet:** Choose for large-scale data analysis, machine learning pipelines, and production environments where storage space and read speed are priorities.

## Task 3: Row-Major vs Column-Major Access Patterns

**Step 1:** Create a large DataFrame with 200,000 rows and 20 numeric columns (random float values). Name the columns `feature_0` through `feature_19`.

**Step 2:** Measure and compare the time for these two operations:

**Operation A — Column access:** Compute the mean of `feature_0` using the pandas column accessor (`df["feature_0"].mean()`).

**Operation B — Row iteration:** Loop through the first 10,000 rows using `df.iloc[i]` and sum the values of `feature_0` manually.

Use `%%timeit` or `%time` to measure each. Record the results.

In [7]:
# Step 1
# 200,000 rows, 20 columns of random floats
data = np.random.rand(200000, 20)
columns = [f"feature_{i}" for i in range(20)]
df_large = pd.DataFrame(data, columns=columns)

# Step 2
print("Timing Column Access (Operation A):")
%timeit -n 100 df_large["feature_0"].mean()

print("\nTiming Row Iteration (Operation B):")
# We only do 10,000 rows because 200,000 would take way too long.
def manual_sum_row_iter(df, n_rows):
    total = 0
    for i in range(n_rows):
        total += df.iloc[i]["feature_0"]
    return total

%time _ = manual_sum_row_iter(df_large, 10000)

Timing Column Access (Operation A):
615 μs ± 117 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)

Timing Row Iteration (Operation B):
CPU times: total: 359 ms
Wall time: 361 ms


**Step 3:** Now convert the DataFrame to a NumPy array and repeat both operations:

**Operation C — NumPy column access:** Compute the mean of column 0 (`arr[:, 0].mean()`).

**Operation D — NumPy row access:** Access the first 10,000 rows sequentially (`arr[i]` in a loop) and sum column 0.

Record the results.

In [8]:
# Convert the DataFrame to a NumPy array
arr = df_large.values

print("Operation C — NumPy Column Access:")
%timeit arr[:, 0].mean()

print("\nOperation D — NumPy Row Access (10,000 rows):")
def numpy_manual_sum(array, n_rows):
    total = 0
    for i in range(n_rows):
        # Accessing the whole row, then the first element
        total += array[i][0]
    return total

%time _ = numpy_manual_sum(arr, 10000)

Operation C — NumPy Column Access:
138 μs ± 7.06 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

Operation D — NumPy Row Access (10,000 rows):
CPU times: total: 0 ns
Wall time: 3.92 ms


**Step 4:** Write a markdown cell that answers:

* Why is column access in pandas so much faster than row iteration?
* Why does the gap narrow (or change) when using NumPy?
* What does this tell you about how pandas and NumPy lay out data in memory?

**Why is column access in pandas so much faster than row iteration?**
Pandas is a **columnar** data structure built on NumPy. In memory, all values for `feature_0` are stored in a single, contiguous block. When we ask for a column mean, the CPU can pull that entire block into its "Cache" all at once. Row iteration (`iloc`) forces the CPU to jump across memory addresses to find each row, then search within that row for the column, which is extremely inefficient.

**Why does the gap narrow (or change) when using NumPy?**
The gap narrows because NumPy is written in highly optimized C code and has much less "overhead" than Pandas. When you call `df.iloc[i]`, Pandas creates a whole new Series object for that row, which is very expensive. NumPy simply points to a memory address, making the row access significantly faster than the Pandas equivalent, even if it’s still not as fast as a vectorized column operation.

**What does this tell you about how pandas and NumPy lay out data in memory?**
This confirms that both libraries prioritize **column-major** (or Fortran-style) layouts for analytical tasks. They are designed for "Vectorized" operations—performing a single math operation on an entire column at once—rather than looping through individual records.

## Task 4: Text vs Binary Format Deep Dive

**Step 1:** Create a DataFrame with 100,000 rows and 5 numeric columns of random float values (use `np.random.seed(42)` for reproducibility). Add one string column called `category` with values randomly chosen from `["alpha", "beta", "gamma"]`.

**Step 2:** Save as CSV and Parquet. Compare file sizes.

In [9]:
# Step 1
np.random.seed(42)
data_task4 = np.random.rand(100000, 5)
df_task4 = pd.DataFrame(data_task4, columns=[f"val_{i}" for i in range(5)])
df_task4['category'] = np.random.choice(["alpha", "beta", "gamma"], 100000)

# Step 2
df_task4.to_csv("task4.csv", index=False)
df_task4.to_parquet("task4.parquet")

# Compare sizes
csv_size = os.path.getsize("task4.csv") / (1024**2) # Size in MB
pq_size = os.path.getsize("task4.parquet") / (1024**2) # Size in MB

print(f"CSV Size: {csv_size:.2f} MB")
print(f"Parquet Size: {pq_size:.2f} MB")

CSV Size: 9.82 MB
Parquet Size: 4.81 MB


**Step 3:** Read the CSV back and check the dtypes. Then read the Parquet back and check the dtypes. Write a markdown cell explaining any differences you observe in how types were preserved.

In [10]:
# Read back
df_from_csv = pd.read_csv("task4.csv")
df_from_parquet = pd.read_parquet("task4.parquet")

print("--- CSV Data Types ---")
print(df_from_csv.dtypes)

print("\n--- Parquet Data Types ---")
print(df_from_parquet.dtypes)

--- CSV Data Types ---
val_0       float64
val_1       float64
val_2       float64
val_3       float64
val_4       float64
category        str
dtype: object

--- Parquet Data Types ---
val_0       float64
val_1       float64
val_2       float64
val_3       float64
val_4       float64
category        str
dtype: object


**Step 4:** Perform a timing comparison: measure how long it takes to read the full dataset from CSV versus from Parquet. Record both times.

In [11]:
print("Timing CSV Read:")
%time _ = pd.read_csv("task4.csv")

print("\nTiming Parquet Read:")
%time _ = pd.read_parquet("task4.parquet")

Timing CSV Read:
CPU times: total: 141 ms
Wall time: 143 ms

Timing Parquet Read:
CPU times: total: 93.8 ms
Wall time: 12.7 ms


**Step 5:** Parquet supports reading only specific columns without loading the entire file. Read only the `category` column from the Parquet file and time it. Compare to reading the full file. Write a markdown cell explaining why this selective read is possible with Parquet but not with CSV.

In [12]:
import pyarrow.parquet as pq

print("Timing Selective Read (Category Column only):")
# We use columns=['category'] to tell the engine to ignore the rest
%time _ = pd.read_parquet("task4.parquet", columns=["category"])

Timing Selective Read (Category Column only):
CPU times: total: 15.6 ms
Wall time: 8.55 ms


**Explain any differences you observe in how types were preserved:**
In the **CSV** file, all schema information is lost; the file is just a stream of characters. While Pandas is good at "guessing" that numbers are floats, complex types like categories or specific datetime precisions often default to generic "objects" or "strings." **Parquet**, however, stores a "metadata schema" at the end of the file. This tells Pandas exactly what the data types are before it even starts reading the rows, ensuring 100% faithful type preservation.

**Why is selective read possible with Parquet but not with CSV?**
Because **CSV** is a row-oriented text format, the computer must scan every single character from the beginning of the file to find where a specific column starts and ends for every row. To get the last column, you effectively have to read the whole file. **Parquet** is a columnar format. It stores data in "chunks" by column. The file metadata includes the exact byte-offset of where each column starts. This allows the system to "jump" straight to the `category` data on the disk and ignore the numeric columns entirely, saving time and memory.

## Task 5: Representation Choices and Trade-Offs

**Step 1:** Create a flat (denormalized) DataFrame representing an online bookstore with at least 12 rows. Include columns: `title`, `author`, `format` (Paperback/E-book), `publisher_name`, `publisher_country`, `price`. Make sure at least two books have multiple formats (so the publisher info is repeated).

In [13]:
# Create the flat version
flat_data = {
    "title": ["The Great Gatsby", "The Great Gatsby", "1984", "1984", "Brave New World", "The Hobbit", "The Hobbit", "Foundation", "Foundation", "Dune", "Dune", "Dune"],
    "author": ["F. Scott Fitzgerald", "F. Scott Fitzgerald", "George Orwell", "George Orwell", "Aldous Huxley", "J.R.R. Tolkien", "J.R.R. Tolkien", "Isaac Asimov", "Isaac Asimov", "Frank Herbert", "Frank Herbert", "Frank Herbert"],
    "format": ["Paperback", "E-book", "Paperback", "E-book", "E-book", "Paperback", "E-book", "Paperback", "E-book", "Paperback", "E-book", "Hardcover"],
    "publisher_name": ["Scribner", "Scribner", "Secker & Warburg", "Secker & Warburg", "Chatto & Windus", "Allen & Unwin", "Allen & Unwin", "Gnome Press", "Gnome Press", "Chilton Books", "Chilton Books", "Chilton Books"],
    "publisher_country": ["USA", "USA", "UK", "UK", "UK", "UK", "UK", "USA", "USA", "USA", "USA", "USA"],
    "price": [15.99, 9.99, 12.50, 7.99, 8.50, 18.00, 10.00, 20.00, 12.00, 25.00, 15.00, 40.00]
}

df_flat = pd.DataFrame(flat_data)
print("Flat DataFrame (Denormalized):")
display(df_flat.head(3))

Flat DataFrame (Denormalized):


,title,author,format,publisher_name,publisher_country,price
0,The Great Gatsby,F. Scott Fitzgerald,Paperback,Scribner,USA,15.99
1,The Great Gatsby,F. Scott Fitzgerald,E-book,Scribner,USA,9.99
2,1984,George Orwell,Paperback,Secker & Warburg,UK,12.50


**Step 2:** Create a normalized version of the same data using two DataFrames:

* `books` with columns: `title`, `author`, `format`, `publisher_id`, `price`
* `publishers` with columns: `publisher_id`, `publisher_name`, `publisher_country`

Verify your normalization by joining the two tables back together and comparing with the flat version.

In [14]:
# Publishers Table
publishers = pd.DataFrame({
    "publisher_id": [1, 2, 3, 4, 5],
    "publisher_name": ["Scribner", "Secker & Warburg", "Chatto & Windus", "Allen & Unwin", "Gnome Press"],
    "publisher_country": ["USA", "UK", "UK", "UK", "USA"]
}).set_index("publisher_id")

# Books Table
books = pd.DataFrame({
    "title": ["The Great Gatsby", "The Great Gatsby", "1984"],
    "author": ["F. Scott Fitzgerald", "F. Scott Fitzgerald", "George Orwell"],
    "format": ["Paperback", "E-book", "Paperback"],
    "publisher_id": [1, 1, 2],
    "price": [15.99, 9.99, 12.50]
})

# Verification via Join
df_joined = books.merge(publishers, on="publisher_id")
print("\nNormalized Verification (Joined):")
display(df_joined.head(3))


Normalized Verification (Joined):


,title,author,format,publisher_id,price,publisher_name,publisher_country
0,The Great Gatsby,F. Scott Fitzgerald,Paperback,1,15.99,Scribner,USA
1,The Great Gatsby,F. Scott Fitzgerald,E-book,1,9.99,Scribner,USA
2,1984,George Orwell,Paperback,2,12.50,Secker & Warburg,UK


**Step 3:** Create a document-oriented version of the same data as a list of dictionaries (JSON-style), where each book is a document containing nested publisher information and a list of editions.

In [15]:
book_documents = [
    {
        "title": "The Great Gatsby",
        "author": "F. Scott Fitzgerald",
        "publisher": {"name": "Scribner", "country": "USA"},
        "editions": [
            {"format": "Paperback", "price": 15.99},
            {"format": "E-book", "price": 9.99}
        ]
    },
    {
        "title": "1984",
        "author": "George Orwell",
        "publisher": {"name": "Secker & Warburg", "country": "UK"},
        "editions": [
            {"format": "Paperback", "price": 12.50},
            {"format": "E-book", "price": 7.99}
        ]
    }
]

**Step 4:** For each representation (flat, normalized, document), write and execute code to answer these three questions:

1. What are all books by a specific author?
2. What is the average price across all editions?
3. Update a publisher's country (simulate the publisher moving to a new country). How many records/documents need to change?

In [16]:
# Flat Representation (DataFrame)
# A. Books by specific author
author_books_flat = df_flat[df_flat['author'] == "George Orwell"]

# B. Average price
avg_price_flat = df_flat['price'].mean()

# C. Update publisher's country (Simulate update)
# How many records change? Every row where the publisher is "Scribner"
rows_to_change = len(df_flat[df_flat['publisher_name'] == "Scribner"])
df_flat.loc[df_flat['publisher_name'] == "Scribner", "publisher_country"] = "Canada"

print(f"Flat - Avg Price: ${avg_price_flat:.2f}")
print(f"Flat - Update 'Scribner' to Canada: {rows_to_change} records updated.")

Flat - Avg Price: $16.25
Flat - Update 'Scribner' to Canada: 2 records updated.


In [17]:
# Normalized Representation (Two Tables)
# A. Books by specific author
author_books_norm = books[books['author'] == "George Orwell"]

# B. Average price
avg_price_norm = books['price'].mean()

# C. Update publisher's country
# How many records change? ONLY 1 record in the publishers table.
publishers.loc[publishers['publisher_name'] == "Scribner", "publisher_country"] = "Canada"
records_updated_norm = 1 

print(f"Normalized - Avg Price: ${avg_price_norm:.2f}")
print(f"Normalized - Update 'Scribner' to Canada: {records_updated_norm} record updated.")

Normalized - Avg Price: $12.83
Normalized - Update 'Scribner' to Canada: 1 record updated.


In [18]:
# Document-Oriented (List of Dicts)
# A. Books by specific author
author_books_doc = [b for b in book_documents if b['author'] == "George Orwell"]

# B. Average price (Need to nested-loop through editions)
all_prices = [edition['price'] for b in book_documents for edition in b['editions']]
avg_price_doc = sum(all_prices) / len(all_prices)

# C. Update publisher's country
# We have to loop through every document to check the publisher name
docs_changed = 0
for b in book_documents:
    if b['publisher']['name'] == "Scribner":
        b['publisher']['country'] = "Canada"
        docs_changed += 1

print(f"Document - Avg Price: ${avg_price_doc:.2f}")
print(f"Document - Update 'Scribner' to Canada: {docs_changed} documents updated.")

Document - Avg Price: $11.62
Document - Update 'Scribner' to Canada: 1 documents updated.


**Step 5:** Write a summary markdown cell comparing the three representations. For each question above, which representation made the answer easiest? Which representation would you choose if the publisher information changes frequently? Which would you choose if each book is always accessed as a standalone record?

**Which representation made the answers easiest?**

* **Author Search:** The **Flat** representation is easiest because you just filter one column without any joins or nested loops.
* **Average Price:** The **Flat** representation is also very fast here for a simple `.mean()`, though the **Normalized** books table is equally simple.
* **Publisher Update:** The **Normalized** version is the clear winner. You only have to change **one record** in the `publishers` table. In the Flat or Document versions, you would have to search and replace the country in every single book record.

**Which would you choose if publisher info changes frequently?**
I would choose the **Normalized** representation. It ensures "Data Integrity"—you only update the information in one place, and the change is automatically reflected across all related books.

**Which would you choose if each book is accessed as a standalone record?**
I would choose the **Document-Oriented** version. Since all the data (editions, publisher, author) is "nested" inside one object, the system can retrieve everything about a book in a single "read" operation without doing expensive joins.